# Tải thư viện

In [1]:
!pip install fastapi uvicorn transformers accelerate omegaconf nest-asyncio

# Tạo file config.yaml

In [2]:
%%writefile config.yaml
model_path: "Qwen/Qwen2.5-3B-Instruct"

Writing config.yaml


# Tạo file model_handler.py


In [3]:
%%writefile model_handler.py
import torch
from omegaconf import OmegaConf
from transformers import AutoTokenizer, AutoModelForCausalLM

class TextGenerator:
    def __init__(self, config_path):
        self.config = OmegaConf.load(config_path)
        self.tokenizer = AutoTokenizer.from_pretrained(self.config.model_path)
        self.model = AutoModelForCausalLM.from_pretrained(
            self.config.model_path,
            device_map="auto",
            torch_dtype="auto"
        )

    def generate_text(self, prompt_text):
        messages = [
            {"role": "system",
            "content": (
            "Bạn là một Chuyên gia Du lịch AI am hiểu sâu sắc về địa lý, văn hóa và ẩm thực của Việt Nam và thế giới. "
            "Nhiệm vụ của bạn là tư vấn du lịch một cách chuyên nghiệp, chính xác và đi thẳng vào vấn đề.\n\n"
            "NGUYÊN TẮC TRẢ LỜI:\n"
            "Trả lời từ 3 đến 4 câu.\n"
            "KHÔNG dùng gạch đầu dòng (-), KHÔNG dùng số thứ tự (1, 2) hoặc tạo danh sách.\n"
            "Dựa hoàn toàn vào kiến thức thực tế. Nếu người dùng hỏi một địa điểm không tồn tại hoặc bạn không biết rõ, hãy trả lời duy nhất 1 câu: 'Xin lỗi, tôi chưa có đủ thông tin chính xác về địa điểm này.'")
            },

            {"role": "user", "content": prompt_text}
        ]

        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

        with torch.no_grad():
            generated_ids = self.model.generate(
                **model_inputs,
                max_new_tokens=512,
                temperature=0.6,
                pad_token_id=self.tokenizer.eos_token_id
            )

        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]

        response = self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

        return response

Writing model_handler.py


# Tạo file main_api.py


In [4]:
%%writefile main_api.py
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from model_handler import TextGenerator

app = FastAPI(title="AI Travel Assistant API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

try:
    generator = TextGenerator("config.yaml")
except Exception as e:
    print(f"Lỗi khi tải model: {e}")


@app.get("/")
async def read_root():
    return {
        "name": "AI Travel Assistant API",
        "description": "API tạo sinh nội dung giới thiệu địa điểm du lịch bằng Qwen2.5."
    }

@app.get("/health")
async def health_check():
    return {"status": "Active", "message": "Hệ thống đang hoạt động bình thường!"}

@app.post("/generate")
async def generate_content(payload: dict):
    prompt_text = payload.get("prompt")
    if not prompt_text or len(prompt_text.strip()) == 0:
        raise HTTPException(status_code=400, detail="Vui lòng nhập câu hỏi hoặc yêu cầu!")

    try:
        result = generator.generate_text(prompt_text)

        return {
            "input_prompt": prompt_text,
            "generated_text": result
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Lỗi hệ thống khi sinh văn bản: {str(e)}")

Writing main_api.py


# Chạy Server và Mở đường hầm Pinggy

In [5]:
import threading
import uvicorn
from main_api import app

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

print("Server started on http://0.0.0.0:8000")

# Mở đường hầm Pinggy: Mở terminal chạy lệnh sau:
# ssh -p 443 -R0:localhost:8000 qr@a.pinggy.io

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Server started on http://0.0.0.0:8000


# Test Local API

In [6]:
import requests

API_URL = "http://127.0.0.1:8000"

print("1. Kiểm tra Endpoint Giới thiệu (GET /)")
response = requests.get(f"{API_URL}/")
print(response.json())
print()

print("2. Kiểm tra Endpoint HEALTH (GET /health)")
response = requests.get(f"{API_URL}/health")
print(response.json())
print()

print("3. Kiểm tra chức năng Tạo Sinh (POST /generate)")
test_cases = [
    {"prompt": "Nội thành Huế có gì thu hút khách du lịch?"},
    {"prompt": "Hãy review chi tiết về khu du lịch trượt tuyết, tuyết rơi quanh năm ở Bến Tre."},
    {"prompt": ""}
]

for i, data in enumerate(test_cases):
    print(f"==== TestCase {i+1}")
    print(f"Prompt: {data['prompt']}")

    response = requests.post(f"{API_URL}/generate", json=data)

    if response.status_code == 200:
        print("AI Trả lời: ", response.json()["generated_text"])
    else:
        print(f"Lỗi {response.status_code}:", response.json())

INFO:     Started server process [1844]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


1. Kiểm tra Endpoint Giới thiệu (GET /)
INFO:     127.0.0.1:38930 - "GET / HTTP/1.1" 200 OK
{'name': 'AI Travel Assistant API', 'description': 'API tạo sinh nội dung giới thiệu địa điểm du lịch bằng Qwen2.5.'}

2. Kiểm tra Endpoint HEALTH (GET /health)
INFO:     127.0.0.1:38940 - "GET /health HTTP/1.1" 200 OK
{'status': 'Active', 'message': 'Hệ thống đang hoạt động bình thường!'}

3. Kiểm tra chức năng Tạo Sinh (POST /generate)
==== TestCase 1
Prompt: Nội thành Huế có gì thu hút khách du lịch?
INFO:     127.0.0.1:38942 - "POST /generate HTTP/1.1" 200 OK
AI Trả lời:  Nội thành Huế thu hút khách du lịch với nhiều di tích lịch sử quan trọng như Kinh thành Huế, Hoàng Thành, các đền chùa như Đền Mẫu Tam Thanh, Đền thờ vua Khải Định. Ngoài ra, con đường Nguyễn Văn Cừ chạy dọc qua nội thành còn nổi tiếng với những hàng quán ẩm thực đặc trưng của Huế.
==== TestCase 2
Prompt: Hãy review chi tiết về khu du lịch trượt tuyết, tuyết rơi quanh năm ở Bến Tre.
INFO:     127.0.0.1:45034 - "POST /genera